# 02 – Modélisation

Cible : `available_t30`, nombre de vélos disponibles 30 min plus tard (régression).

## 1. Chargement du fichier

Fichier nettoyé, exporté par `01-exploration.ipynb`.

In [17]:
from pathlib import Path
import pandas as pd

path = Path("raw/data/historique_nettoye.csv")

In [18]:
df = pd.read_csv(path, dtype={"station_code": "str"}, parse_dates=["horodatage"])
df.shape

(2870500, 24)

In [19]:
df.head()

,horodatage,station_code,capacity,velo_meca,velo_elec,station_name,operative,latitude,longitude,available,...,heure_cos,is_weekend,heure,delta_abs,is_rush_hour,temperature_2m,wind_speed_10m,precipitation,rain,weather_label
0,2026-10-16 13:20:00+00:00,1001,17,2,5,Quai de l'Horloge - Pont Neuf,True,48.85706,2.3418,7,...,-0.939693,False,15,7,False,21.50,13.49,0.0,0.0,Overcast
1,2026-10-16 13:25:00+00:00,1001,17,2,5,Quai de l'Horloge - Pont Neuf,True,48.85706,2.3418,7,...,-0.932008,False,15,6,False,21.49,13.42,0.0,0.0,Overcast
2,2026-10-16 13:35:00+00:00,1001,17,7,7,Quai de l'Horloge - Pont Neuf,True,48.85706,2.3418,14,...,-0.915311,False,15,1,False,21.45,13.19,0.0,0.0,Overcast
3,2026-10-16 13:40:00+00:00,1001,17,7,7,Quai de l'Horloge - Pont Neuf,True,48.85706,2.3418,14,...,-0.906308,False,15,1,False,21.44,13.03,0.0,0.0,Overcast
4,2026-10-16 13:45:00+00:00,1001,17,7,7,Quai de l'Horloge - Pont Neuf,True,48.85706,2.3418,14,...,-0.896873,False,15,1,False,21.42,12.86,0.0,0.0,Overcast


## 2. Découpage train / test

Les 2 derniers jours du jeu nettoyé forment le jeu de test : on n'y touche pas avant l'évaluation finale. Le reste constitue le jeu d'entraînement, sur lequel la validation et le réglage des modèles se font.

In [20]:
# format 17-10-2026 18:30
horodatage = df["horodatage"].astype(str).str.replace(
    r"(\d{4})-(\d{2})-(\d{2}) (\d{2}):(\d{2}).*", r"\3-\2-\1 \4:\5", regex=True
)
df["horodatage"] = pd.to_datetime(horodatage, format="%d-%m-%Y %H:%M")

debut_test = pd.to_datetime("24-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [21]:
test = df[df["horodatage"] >= debut_test]
train = df[df["horodatage"] < debut_test]
print(train.shape, test.shape)


(2268356, 24) (602144, 24)


In [22]:
ratio = train.shape[0] / df.shape[0]
print(ratio)

0.7902302734715206


## 3. Plis d'entraînement

On travaille uniquement dans `train`. Chaque pli teste sur un jour et s'entraîne sur tous les jours d'avant :

- pli 1 : entraînement du 16 au 17, validation le 18
- pli 2 : entraînement du 16 au 18, validation le 19
- et ainsi de suite jusqu'au pli 6 (validation le 23)

Le 16 et le 17 sont des journées incomplètes (début de la collecte, relevés irréguliers) mais on les garde pour l'entraînement. On retire 30 minutes à la fin de l'entraînement : la cible d'une ligne est observée 30 minutes plus tard, elle ne doit pas tomber dans le jour de validation.

In [23]:
purge = pd.Timedelta(minutes=30)

In [24]:
j16 = pd.to_datetime("16-10-2026 00:00", format="%d-%m-%Y %H:%M")
j17 = pd.to_datetime("17-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [25]:
j18 = pd.to_datetime("18-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [26]:
j19 = pd.to_datetime("19-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [27]:
j20 = pd.to_datetime("20-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [28]:
j21 = pd.to_datetime("21-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [29]:
j22 = pd.to_datetime("22-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [30]:
j23 = pd.to_datetime("23-10-2026 00:00", format="%d-%m-%Y %H:%M")

In [31]:
j24 = pd.to_datetime("24-10-2026 00:00", format="%d-%m-%Y %H:%M")

### Pli 1

In [32]:
train_1 = train[(train["horodatage"] >= j16) & (train["horodatage"] < j18 - purge)]
val_1 = train[(train["horodatage"] >= j18) & (train["horodatage"] < j19)]
print(train_1.shape, val_1.shape)

(188325, 24) (346275, 24)


### Pli 2

In [33]:
train_2 = train[(train["horodatage"] >= j16) & (train["horodatage"] < j19 - purge)]
val_2 = train[(train["horodatage"] >= j19) & (train["horodatage"] < j20)]
print(train_2.shape, val_2.shape)

(534600, 24) (343845, 24)


### Pli 3

In [34]:
train_3 = train[(train["horodatage"] >= j16) & (train["horodatage"] < j20 - purge)]
val_3 = train[(train["horodatage"] >= j20) & (train["horodatage"] < j21)]
print(train_3.shape, val_3.shape)

(877230, 24) (348337, 24)


### Pli 4

In [35]:
train_4 = train[(train["horodatage"] >= j16) & (train["horodatage"] < j21 - purge)]
val_4 = train[(train["horodatage"] >= j21) & (train["horodatage"] < j22)]
print(train_4.shape, val_4.shape)

(1226792, 24) (348295, 24)


### Pli 5

In [36]:
train_5 = train[(train["horodatage"] >= j16) & (train["horodatage"] < j22 - purge)]
val_5 = train[(train["horodatage"] >= j22) & (train["horodatage"] < j23)]
print(train_5.shape, val_5.shape)

(1573862, 24) (337572, 24)


### Pli 6

In [37]:
train_6 = train[(train["horodatage"] >= j16) & (train["horodatage"] < j23 - purge)]
val_6 = train[(train["horodatage"] >= j23) & (train["horodatage"] < j24)]
print(train_6.shape, val_6.shape)

(1911440, 24) (349632, 24)
